**Question 1**

In [1]:
import pandas as pd
import requests
from io import StringIO
import re

In [ ]:
# Data Loading (from iposcoop.com)

url = "https://www.iposcoop.com/ipos-recently-filed"
 
headers = {
    'User-Agent': (
        'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) '
        'AppleWebKit/537.36 (KHTML, like Gecko) '
        'Chrome/91.0.4472.124 Safari/537.36'
    )
}
 
try:
    response = requests.get(url, headers=headers)
    response.raise_for_status()
    
    # Use StringIO buffer
    df_ipos = pd.read_html(StringIO(response.text))[0]
    print(f"Successfully loaded IPO data. Shape: {df_ipos.shape}")
    print(f"\nColumn names:\n{df_ipos.columns.tolist()}\n")
    
except Exception as e:
    print(f"Error loading data: {e}")
    exit()

Successfully loaded IPO data. Shape: (500, 10)

Column names:
['File Date', 'Company', 'Symbol', 'Managers', 'Shares (millions)', 'Price Low', 'Price High', 'Est $ Vol (millions)', 'Expected To Trade', 'SCOOP Rating']



In [ ]:
# filter withdrawn IPOs

withdrawn_df = df_ipos[df_ipos['Expected To Trade'] == 'Withdrawn'].copy()
print(f"Number of withdrawn IPOs: {len(withdrawn_df)}")
withdrawn_df.head()

Number of withdrawn IPOs: 34


,File Date,Company,Symbol,Managers,Shares (millions),Price Low,Price High,Est $ Vol (millions),Expected To Trade,SCOOP Rating
17,2026-09-18,Rise Smart Group (Withdrawn),RSHL,American Trust Investment Services/Prime Numbe...,1.80,$4.00,$4.00,$7.20,Withdrawn,S/O
20,2026-09-17,C2 Capital Group (Withdrawn),CCLV,Dominari Securities,3.75,$4.00,$5.00,$16.88,Withdrawn,S/O
32,2026-09-15,OTSAW Ltd. (Withdrawn),OTSA,Aegis Capital,4.40,$4.50,$5.50,$22.00,Withdrawn,S/O
38,2026-09-10,Motive Technologies (Withdrawn),MTVE,J.P.Morgan/Citigroup/Barclays/Jefferies/RBC Ca...,0.00,NaN,NaN,$100.00,Withdrawn,S/O
43,2026-09-04,Idea Tech Holding (Withdrawn),IDTL,R.F. Lafferty & Co.,2.00,$4.00,$5.00,$9.00,Withdrawn,S/O


In [ ]:
# Company Classification

def classify_company_type(company_name):
    """
    Categorize company based on patterns (order matters).
    Returns the first matched category.
    """
    if pd.isna(company_name):
        return 'Other'
    
    name = str(company_name)
    
    # Order matters: check in this sequence
    if 'Technologies' in name:
        return 'Technologies'
    elif 'Acquisition Corp' in name or 'Acquisition Corporation' in name or 'Corp' in name:
        return 'Acquisition Corp'
    elif 'Inc' in name or 'Incorporated' in name:
        return 'Inc.'
    elif 'Group' in name:
        return 'Group'
    elif 'Ltd' in name or 'Limited' in name:
        return 'Limited'
    elif 'Holdings' in name or 'Holding' in name:
        return 'Holdings'
    else:
        return 'Other'
 
withdrawn_df['Company Type'] = withdrawn_df['Company'].apply(classify_company_type)
print(f"\nCompany Type distribution:")
print(withdrawn_df['Company Type'].value_counts())


Company Type distribution:
Company Type
Limited             8
Other               8
Group               5
Acquisition Corp    5
Holdings            4
Technologies        3
Inc.                1
Name: count, dtype: int64


In [11]:
# Price Parsing

def extract_numeric(price_str):
    """
    Extract numeric value from strings like '$8.00' or '-'
    Returns float or NaN for missing values
    """
    if pd.isna(price_str):
        return None
    
    price_str = str(price_str).strip()
    
    # Handle missing values
    if price_str == '-' or price_str == '':
        return None
    
    # Remove all non-numeric characters except decimal point
    cleaned = re.sub(r'[^\d.]', '', price_str)
    
    try:
        return float(cleaned) if cleaned else None
    except ValueError:
        return None

withdrawn_df['Price_Low'] = withdrawn_df['Price Low'].apply(extract_numeric)
withdrawn_df['Price_High'] = withdrawn_df['Price High'].apply(extract_numeric)
withdrawn_df['Avg_price'] = (withdrawn_df['Price_Low'] + withdrawn_df['Price_High']) / 2.0

withdrawn_df[['Company', 'Price Low', 'Price High', 'Avg_price']].head(50)

,Company,Price Low,Price High,Avg_price
17,Rise Smart Group (Withdrawn),$4.00,$4.00,4.00
20,C2 Capital Group (Withdrawn),$4.00,$5.00,4.50
32,OTSAW Ltd. (Withdrawn),$4.50,$5.50,5.00
38,Motive Technologies (Withdrawn),NaN,NaN,NaN
43,Idea Tech Holding (Withdrawn),$4.00,$5.00,4.50
50,Hornbeck Offshore Services (Withdrawn),NaN,NaN,NaN
52,Coolbit Technologies Ltd. (Withdrawn),$4.00,$5.00,4.50
59,Timber Road Acquisition Corp. (Withdrawn),$10.00,$10.00,10.00
69,pan-Africa Corp. (Withdrawn),$10.00,$10.00,10.00
84,Living Homeopathy (Withdrawn),$4.00,$6.00,5.00


In [15]:
# Numeric Conversion

withdrawn_df['Shares_millions'] = pd.to_numeric(
    withdrawn_df['Shares (millions)'].astype(str).str.replace('$', '').str.replace(',', ''),
    errors='coerce'
)

withdrawn_df['Est_Vol_millions'] = pd.to_numeric(
    withdrawn_df['Est $ Vol (millions)'].astype(str).str.replace('$', '').str.replace(',', ''),
    errors='coerce'
)

In [16]:
# Value Calculation

withdrawn_df['Shares_offered_value'] = (withdrawn_df['Shares_millions'] * withdrawn_df['Avg_price']).fillna(withdrawn_df['Est_Vol_millions'])

In [17]:
# Aggregation
result = withdrawn_df.groupby('Company Type')['Shares_offered_value'].sum().sort_values(ascending=False)

print("="*60)
print("WITHDRAWAL VALUES BY COMPANY TYPE (in $ millions)")
print("="*60)
print(result)
print()
print(f"Answer: {result.idxmax()} - ${result.max():,.2f} million")

WITHDRAWAL VALUES BY COMPANY TYPE (in $ millions)
Company Type
Acquisition Corp    499.9850
Inc.                351.0000
Holdings            311.6575
Other               290.4450
Limited             219.2500
Technologies        184.9000
Group                56.5750
Name: Shares_offered_value, dtype: float64

Answer: Acquisition Corp - $499.99 million


**Qeustion 2**

In [1]:
import pandas as pd
import numpy as np
import yfinance as yf
from io import StringIO
import requests

In [ ]:
# Data Loading

url = "https://www.iposcoop.com/2025-pricings/"

headers = {'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36'}
response = requests.get(url, headers=headers)
df_ipos_2025 = pd.read_html(StringIO(response.text))[0]

print(f"Total IPOs 2025: {len(df_ipos_2025)}")

Total IPOs 2025: 231


In [3]:
# Filtering

df_ipos_2025['Offer Date'] = pd.to_datetime(df_ipos_2025['Offer Date'], errors='coerce')
df_ipos_2025['Return'] = pd.to_numeric(df_ipos_2025['Return'].astype(str).str.replace('%', ''), errors='coerce')

filtered_ipos = df_ipos_2025[
    (df_ipos_2025['Offer Date'] < '2025-09-01') & 
    (df_ipos_2025['Return'] != 0)
].copy()

print(f"Filtered IPOs (before Sept 1, 2025, non-zero return): {len(filtered_ipos)}")
print(filtered_ipos[['Symbol', 'Offer Date', 'Return']].head())

Filtered IPOs (before Sept 1, 2025, non-zero return): 146
   Symbol Offer Date  Return
68    PMI 2025-08-29  -58.00
69   TGHL 2025-08-28  -90.00
70    THH 2025-08-28  636.75
71   CURX 2025-08-26  -90.75
72   CEPF 2025-08-21    1.20


In [22]:
# Data Download

tickers = filtered_ipos['Symbol'].tolist()
stocks_df = pd.DataFrame()

for ticker in tickers:
    try:
        data = yf.download(ticker, start='2024-09-01', end='2026-09-12', progress=False)
        if len(data) > 0:
            # Extract Close column for this ticker
            close_data = data[('Close', ticker)].reset_index()
            close_data.columns = ['Date', 'Close']
            close_data['Ticker'] = ticker
            stocks_df = pd.concat([stocks_df, close_data], ignore_index=True)
    except:
        pass

print(f"Downloaded stocks: {stocks_df['Ticker'].nunique()}")
print(stocks_df.head())

$MJID: possibly delisted; no timezone found

1 Failed download:
['MJID']: possibly delisted; no timezone found
$EMPG: possibly delisted; no timezone found

1 Failed download:
['EMPG']: possibly delisted; no timezone found
$CAEP: possibly delisted; no timezone found

1 Failed download:
['CAEP']: possibly delisted; no timezone found
$PTNM: possibly delisted; no timezone found

1 Failed download:
['PTNM']: possibly delisted; no timezone found
$AHL: possibly delisted; no timezone found

1 Failed download:
['AHL']: possibly delisted; no timezone found
$CEPT: possibly delisted; no timezone found

1 Failed download:
['CEPT']: possibly delisted; no timezone found
$SDM: possibly delisted; no timezone found

1 Failed download:
['SDM']: possibly delisted; no timezone found
$TBH: possibly delisted; no timezone found

1 Failed download:
['TBH']: possibly delisted; no timezone found
$WGRX: possibly delisted; no timezone found

1 Failed download:
['WGRX']: possibly delisted; no timezone found
$AGH: p

Downloaded stocks: 132
        Date  Close Ticker
0 2025-08-29  229.5    PMI
1 2025-09-02  257.0    PMI
2 2025-09-03  264.5    PMI
3 2025-09-04  286.5    PMI
4 2025-09-05  292.5    PMI


In [28]:
# Feature Engineering

stocks_df.columns.name = None

stocks_df['Close'] = pd.to_numeric(stocks_df['Close'], errors='coerce')
stocks_df['growth_252d'] = stocks_df.groupby('Ticker')['Close'].transform(lambda x: x / x.shift(252))
stocks_df['volatility'] = stocks_df.groupby('Ticker')['Close'].transform(lambda x: x.rolling(30).std() * np.sqrt(252))

print(stocks_df[['Ticker', 'growth_252d', 'volatility']].tail(10))

      Ticker  growth_252d  volatility
45231   INLF     0.000998   14.494399
45232   INLF     0.001050   13.188799
45233   INLF     0.001095   11.542300
45234   INLF     0.001044   11.471029
45235   INLF     0.001150   11.510758
45236   INLF     0.001312   11.061412
45237   INLF     0.001334   10.745782
45238   INLF     0.001288   10.799118
45239   INLF     0.001195   11.244236
45240   INLF     0.001005   11.457569


In [27]:
# Sharpe Ratio Calculation

stocks_df['Sharpe'] = (stocks_df['growth_252d'] - 0.05) / stocks_df['volatility']

print(stocks_df[['Ticker', 'growth_252d', 'volatility', 'Sharpe']].tail())

      Ticker  growth_252d  volatility    Sharpe
45236   INLF     0.001312   11.061412 -0.004402
45237   INLF     0.001334   10.745782 -0.004529
45238   INLF     0.001288   10.799118 -0.004511
45239   INLF     0.001195   11.244236 -0.004340
45240   INLF     0.001005   11.457569 -0.004276


In [29]:
# Final Analysis

stocks_df['Date'] = pd.to_datetime(stocks_df['Date'])
final_df = stocks_df[stocks_df['Date'] == '2026-09-11'].copy()

print(f"\nStocks on 2026-09-11: {len(final_df)}")
print("\nSharpe Ratio statistics:")
print(final_df['Sharpe'].describe())
print(f"\nMedian Sharpe Ratio: {final_df['Sharpe'].median()}")


Stocks on 2026-09-11: 132

Sharpe Ratio statistics:
count    130.000000
mean            inf
std             NaN
min       -0.040147
25%        0.012042
50%        0.050142
75%        0.133619
max             inf
Name: Sharpe, dtype: float64

Median Sharpe Ratio: 0.050142446812154136


/Users/rob/Projects/GitHub/stock-markets-analytics-zoomcamp/.venv/lib/python3.12/site-packages/pandas/core/nanops.py:1028: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)


In [30]:
# Compare median vs. mean for growth_252d
print("Growth (252d) Statistics:")
print(f"Mean growth_252d: {final_df['growth_252d'].mean():.4f}")
print(f"Median growth_252d: {final_df['growth_252d'].median():.4f}")
print(f"Difference (mean - median): {final_df['growth_252d'].mean() - final_df['growth_252d'].median():.4f}")
print(f"Max growth: {final_df['growth_252d'].max():.4f}")
print(f"Min growth: {final_df['growth_252d'].min():.4f}")

Growth (252d) Statistics:
Mean growth_252d: 1.0576
Median growth_252d: 0.5945
Difference (mean - median): 0.4631
Max growth: 33.6383
Min growth: 0.0010


There's significant positive skew in growth returns, with the mean (1.06x) substantially higher than the median (0.59x), a difference of 0.46. This indicates that a handful of high-growth outliers are pulling the average well above what a typical IPO experienced. The maximum growth of 33.64x (MGRT) significantly inflates the mean, suggesting that most IPOs performed more modestly.

In [31]:
# Count of stocks that reached 252-day milestone
stocks_with_252d = final_df['growth_252d'].notna().sum()
print(f"\nStocks with 252-day data: {stocks_with_252d} out of {len(final_df)}")


Stocks with 252-day data: 130 out of 132


Nearly all stocks reached the 252-day trading milestone, with 130 out of 132 stocks (98.5%) successfully accumulating a full year of price data. Only 2 stocks delisted before completing the period.

In [32]:
# Compare Sharpe ratio vs growth

comparison = final_df[['Ticker', 'growth_252d', 'Sharpe']].dropna().sort_values('Sharpe', ascending=False)
print("\nTop 10 by Sharpe Ratio:")
print(comparison.head(10))
print("\nTop 10 by Growth:")
print(comparison.sort_values('growth_252d', ascending=False).head(10))


Top 10 by Sharpe Ratio:
      Ticker  growth_252d        Sharpe
12383   MAMK     6.532663           inf
5286    EFTY     2.106592           inf
4228    MAGH     5.451613  4.204751e+06
6669   HCMAU     1.034585  5.385326e+00
1311    CEPF     1.024233  2.770994e+00
20735   TMDE     0.776166  1.751178e+00
12084   TLIH     0.989673  1.448889e+00
44395   CEPO     1.031160  1.233763e+00
21793   HXHX     0.409565  7.532461e-01
36998   FBGL     0.576923  6.287732e-01

Top 10 by Growth:
      Ticker  growth_252d        Sharpe
10385   MGRT    33.638270  2.561722e-01
12383   MAMK     6.532663           inf
2869    BUUU     5.930931  8.426321e-02
4228    MAGH     5.451613  4.204751e+06
10893    ALM     3.480899  9.425795e-02
35797   XHLD     2.449658  4.528417e-02
3411    RYOJ     2.268293  3.082141e-01
5286    EFTY     2.106592           inf
14832   SLDE     1.852848  7.359298e-02
12684     CV     1.818182  2.617637e-01


Risk-adjusted returns (Sharpe ratio) do not track with pure growth metrics. The highest-growth stock (MGRT with 33.64x returns) has a relatively poor Sharpe ratio of 0.26, indicating high volatility relative to returns. Conversely, stocks with the highest Sharpe ratios often show more modest growth rates. For example, HCMAU achieved only 1.03x growth but a Sharpe ratio of 5.39, indicating much better risk-adjusted performance. This demonstrates that controlling for volatility reveals a different ranking of stocks than growth alone, and that lower-volatility positions can deliver more attractive risk-adjusted returns even with lower nominal growth.

**Question 3**

In [33]:
import pandas as pd
import numpy as np
import yfinance as yf
from io import StringIO
import requests

In [34]:
# Data Loading

url = "https://www.iposcoop.com/2025-pricings/"

headers = {'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36'}
response = requests.get(url, headers=headers)
df_ipos_2025 = pd.read_html(StringIO(response.text))[0]

print(f"Total IPOs 2025: {len(df_ipos_2025)}")

# Filtering

df_ipos_2025['Offer Date'] = pd.to_datetime(df_ipos_2025['Offer Date'], errors='coerce')
df_ipos_2025['Return'] = pd.to_numeric(df_ipos_2025['Return'].astype(str).str.replace('%', ''), errors='coerce')

filtered_ipos = df_ipos_2025[
    (df_ipos_2025['Offer Date'] < '2025-09-01') & 
    (df_ipos_2025['Return'] != 0)
].copy()

print(f"Filtered IPOs (before Sept 1, 2025, non-zero return): {len(filtered_ipos)}")
print(filtered_ipos[['Symbol', 'Offer Date', 'Return']].head())

# Data Download

tickers = filtered_ipos['Symbol'].tolist()
stocks_df = pd.DataFrame()

for ticker in tickers:
    try:
        data = yf.download(ticker, start='2024-09-01', end='2026-09-12', progress=False)
        if len(data) > 0:
            # Extract Close column for this ticker
            close_data = data[('Close', ticker)].reset_index()
            close_data.columns = ['Date', 'Close']
            close_data['Ticker'] = ticker
            stocks_df = pd.concat([stocks_df, close_data], ignore_index=True)
    except:
        pass

print(f"Downloaded stocks: {stocks_df['Ticker'].nunique()}")
print(stocks_df.head())

Total IPOs 2025: 231
Filtered IPOs (before Sept 1, 2025, non-zero return): 146
   Symbol Offer Date  Return
68    PMI 2025-08-29  -58.00
69   TGHL 2025-08-28  -90.00
70    THH 2025-08-28  636.75
71   CURX 2025-08-26  -90.75
72   CEPF 2025-08-21    1.20


$MJID: possibly delisted; no timezone found

1 Failed download:
['MJID']: possibly delisted; no timezone found
$EMPG: possibly delisted; no timezone found

1 Failed download:
['EMPG']: possibly delisted; no timezone found
$CAEP: possibly delisted; no timezone found

1 Failed download:
['CAEP']: possibly delisted; no timezone found
$PTNM: possibly delisted; no timezone found

1 Failed download:
['PTNM']: possibly delisted; no timezone found
$AHL: possibly delisted; no timezone found

1 Failed download:
['AHL']: possibly delisted; no timezone found
$CEPT: possibly delisted; no timezone found

1 Failed download:
['CEPT']: possibly delisted; no timezone found
$SDM: possibly delisted; no timezone found

1 Failed download:
['SDM']: possibly delisted; no timezone found
$TBH: possibly delisted; no timezone found

1 Failed download:
['TBH']: possibly delisted; no timezone found
$WGRX: possibly delisted; no timezone found

1 Failed download:
['WGRX']: possibly delisted; no timezone found
$AGH: p

Downloaded stocks: 132
        Date  Close Ticker
0 2025-08-29  229.5    PMI
1 2025-09-02  257.0    PMI
2 2025-09-03  264.5    PMI
3 2025-09-04  286.5    PMI
4 2025-09-05  292.5    PMI


In [35]:
# Calculate future growth for 1-12 months (21 trading days per month)
for month in range(1, 13):
    days = month * 21
    stocks_df[f'future_growth_{month}_m'] = stocks_df.groupby('Ticker')['Close'].transform(
        lambda x: x.shift(-days) / x
    )

print("Future growth columns created")
print(stocks_df.columns)

Future growth columns created
Index(['Date', 'Close', 'Ticker', 'future_growth_1_m', 'future_growth_2_m',
       'future_growth_3_m', 'future_growth_4_m', 'future_growth_5_m',
       'future_growth_6_m', 'future_growth_7_m', 'future_growth_8_m',
       'future_growth_9_m', 'future_growth_10_m', 'future_growth_11_m',
       'future_growth_12_m'],
      dtype='str')


In [36]:
# Identify IPO entry points (first trading day for each ticker)
ipo_entry = stocks_df.groupby('Ticker').agg({
    'Date': 'min'
}).reset_index()
ipo_entry.columns = ['Ticker', 'IPO_Date']

print(f"IPO entry points identified for {len(ipo_entry)} stocks")
print(ipo_entry.head())

IPO entry points identified for 132 stocks
  Ticker   IPO_Date
0   AAPG 2025-01-24
1   AARD 2025-02-13
2   ADVB 2025-03-06
3    AII 2025-05-08
4   AIRO 2025-06-13


In [37]:
# Join to get returns from IPO date
ipo_returns = stocks_df.merge(
    ipo_entry,
    left_on=['Ticker', 'Date'],
    right_on=['Ticker', 'IPO_Date'],
    how='inner'
)

print(f"IPO return records: {len(ipo_returns)}")
print(ipo_returns.head())

IPO return records: 132
        Date       Close Ticker  future_growth_1_m  future_growth_2_m  \
0 2025-08-29  229.500000    PMI           1.851852           0.714597   
1 2025-08-28    3.380000   TGHL           0.491124           0.235503   
2 2025-08-27  241.899994    THH           0.300951           0.367921   
3 2025-08-26   80.400002   CURX           0.168905           0.164428   
4 2025-08-21   10.120000   CEPF           1.004941           1.038538   

   future_growth_3_m  future_growth_4_m  future_growth_5_m  future_growth_6_m  \
0           0.464052           0.381264           0.381264           0.209150   
1           0.157988           0.108580           0.195266           0.119231   
2           0.431997           0.700083           0.032245           0.022075   
3           0.115920           0.099502           0.087562           0.114428   
4           1.023715           1.017787           1.032609           1.031621   

   future_growth_7_m  future_growth_8_m  future_gr

In [38]:
# Calculate median growth for each holding period
growth_cols = [f'future_growth_{m}_m' for m in range(1, 13)]
stats = ipo_returns[growth_cols].describe()

print("\nGrowth Statistics by Holding Period:")
print(stats)

# Find optimal month
median_row = ipo_returns[growth_cols].median()
optimal_month = int(median_row.idxmax().split('_')[2])
optimal_value = median_row.max()

print(f"\nOptimal holding period: {optimal_month} months")
print(f"Median growth: {optimal_value:.4f}x")


Growth Statistics by Holding Period:
       future_growth_1_m  future_growth_2_m  future_growth_3_m  \
count         132.000000         131.000000         131.000000   
mean            1.052528           1.176740           1.151785   
std             1.019355           1.732228           1.738123   
min             0.072941           0.106197           0.090147   
25%             0.708835           0.584863           0.442477   
50%             0.929540           0.889891           0.783198   
75%             1.094558           1.148836           1.098424   
max             8.787346          16.066783          16.344464   

       future_growth_4_m  future_growth_5_m  future_growth_6_m  \
count         131.000000         131.000000         131.000000   
mean            1.037548          77.327609          54.471764   
std             1.329384         873.617580         611.503340   
min             0.063927           0.032245           0.022075   
25%             0.392759           0.

The analysis reveals a clear and consistent downward trend in median returns as the holding period extends from 1 to 12 months. The optimal holding period is 1 month, with a median growth of 0.9295x, representing a nearly 7% loss for the typical IPO holder. Each subsequent month sees median returns decline further, bottoming at 0.4918x (approximately 51% loss) at the 12-month mark. This suggests that most 2025 IPOs underperformed significantly in their first year of trading.

In [39]:
# Comparison: Median vs Mean
print("\nMedian vs Mean by Month:")
comparison = pd.DataFrame({
    'Month': range(1, 13),
    'Median': [ipo_returns[f'future_growth_{m}_m'].median() for m in range(1, 13)],
    'Mean': [ipo_returns[f'future_growth_{m}_m'].mean() for m in range(1, 13)]
})
comparison['Difference'] = comparison['Mean'] - comparison['Median']
print(comparison)


Median vs Mean by Month:
    Month    Median       Mean  Difference
0       1  0.929540   1.052528    0.122988
1       2  0.889891   1.176740    0.286850
2       3  0.783198   1.151785    0.368587
3       4  0.741678   1.037548    0.295871
4       5  0.700796  77.327609   76.626812
5       6  0.753319  54.471764   53.718445
6       7  0.673944  41.140472   40.466528
7       8  0.603508  18.276712   17.673203
8       9  0.580294  43.709719   43.129425
9      10  0.533333  51.525280   50.991946
10     11  0.481811  47.584499   47.102688
11     12  0.491838  17.466811   16.974972


While median returns decay steadily from month 1 to month 12, the mean values become increasingly distorted by extreme outliers starting in month 5, with the mean reaching 77x in month 5 and 51x in month 10, while the corresponding medians remain around 0.7x and 0.53x respectively. These astronomical mean values indicate the presence of a small number of extreme gainers (likely delisted or bankrupt stocks showing data anomalies) that completely distort the average.